In [9]:
import pandas as pd
import geopandas as gpd
import json
import os
from shapely.geometry import Point

# =========================================================
# 1. PATH SETUP
# =========================================================
TRACT_SHAPEFILE_PATH = "../data/raw/census/tl_2023_36_tract.shp"
ADVAN_PATH           = "../data/raw/advan"
ADVAN_FILES          = [os.path.join(ADVAN_PATH, f) for f in os.listdir(ADVAN_PATH) if f.endswith(".csv")]
OUTPUT_FILE          = "../data/intermediate/park_flows_placekey.csv"


In [10]:
# =========================================================
# 2. PREPARE GEOGRAPHY (NYC PROJECTION)
# =========================================================
print("Preparing NYC geography...")
tracts = gpd.read_file(TRACT_SHAPEFILE_PATH)
nyc_counties = ['005', '047', '061', '081', '085']
nyc_tracts = tracts[tracts['COUNTYFP'].isin(nyc_counties)].copy()

# Project to NYC State Plane (Feet) for accurate Euclidean distance
nyc_tracts = nyc_tracts.to_crs("EPSG:2263")
nyc_tracts['tract_centroid'] = nyc_tracts.geometry.centroid
tract_lookup = nyc_tracts[['GEOID', 'tract_centroid']].set_index('GEOID')


Preparing NYC geography...


In [11]:
# =========================================================
# 3. BUILD FLOW RECORDS
# Distance is recorded but no filter is applied here.
# The 10km filter is applied at the property level in 05.
# =========================================================
all_flows = []

for file_path in ADVAN_FILES:
    print(f"Reading: {os.path.basename(file_path)}")

    df = pd.read_csv(file_path, usecols=['PLACEKEY', 'LATITUDE', 'LONGITUDE', 'VISITOR_HOME_CBGS'])
    df = df.dropna(subset=['VISITOR_HOME_CBGS'])
    df = df[df['VISITOR_HOME_CBGS'].str.strip() != '']

    geometry = [Point(xy) for xy in zip(df['LONGITUDE'], df['LATITUDE'])]
    gdf_parks = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326").to_crs("EPSG:2263")

    for _, row in gdf_parks.iterrows():
        park_id = row['PLACEKEY']

        try:
            home_dict = json.loads(row['VISITOR_HOME_CBGS'])
            if isinstance(home_dict, str):
                home_dict = json.loads(home_dict)
        except (json.JSONDecodeError, TypeError):
            continue

        for cbg_id, visits in home_dict.items():
            tract_id = str(cbg_id)[:11]

            if tract_id in tract_lookup.index:
                tract_centroid = tract_lookup.loc[tract_id, 'tract_centroid']
                distance_ft = row['geometry'].distance(tract_centroid)
                all_flows.append({
                    'tract_i':     tract_id,
                    'park_j':      park_id,
                    'visits':      visits,
                    'distance_km': round((distance_ft * 0.3048) / 1000, 3),
                })

print(f"Total flow records captured: {len(all_flows)}")


Reading: 2022-04--data_01c238ce-0307-b001-0042-fa0705e531da_906_6_0_unzipped.csv
Reading: 2022-05--data_01c238ce-0307-b001-0042-fa0705e531da_806_4_0_unzipped.csv
Reading: 2022-06--data_01c238ce-0307-b001-0042-fa0705e531da_1406_1_0_unzipped.csv
Total flow records captured: 533891


In [12]:
# =========================================================
# 4. AGGREGATE AND SAVE
# =========================================================
print("Aggregating flows...")
final_df = pd.DataFrame(all_flows)
final_df = final_df.groupby(['tract_i', 'park_j', 'distance_km'], as_index=False)['visits'].sum()

final_df = final_df.sort_values(['tract_i', 'park_j']).reset_index(drop=True)
final_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(final_df)} rows to: {OUTPUT_FILE}")


Aggregating flows...
Saved 216367 rows to: ../data/intermediate/park_flows_placekey.csv
